This notebook shows a common use case of DataPrep, such as:

Feature Visualization: uses create_report().show_browser to show distribution of all columns (mean, std dev, hist)
Missing Value Handling: automatically reports the number of missing cells in each column within the report 
Plotting: uses .plot() from dataprep.eda to create quick plots of data


Workflow Description
In this notebook, we are using the California Housing Dataset from Scikit-Learn. First the data is fetched then converted to a dataframe for feature engineering and model deployment. The models in this example are LinearRegression and RandomForest. DataPrep was heavily used to get a better understanding of features and inform ways to improve model performance. 

The 2 cells below import the required libraries and install Dataprep for execution. Please add your API key as required.

In [1]:
!pip install dataprep numpy pandas matplotlib scikit-learn scipy seaborn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 10.0 MB/s  0:00:00eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [20 lines of output]
      Traceback (most recent call last):
        File "/usr/local/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
        File "/usr/local/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                   

In [ ]:
#get california housing data from sklearn
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(as_frame=True)
print(housing.data.shape, housing.target.shape)


In [ ]:
type(housing) #need to conv to dF

In [ ]:
import pandas as pd
import numpy as np
from dataprep.eda import create_report

df = housing.frame
create_report(df).show_browser()  #opens report in another Tab

In [ ]:
df.info()

In [ ]:
df.head(10) 

In [ ]:
df.columns

In [ ]:
#Feature engineer (only run once)     
#DataPrep signal shows weak correlations, combining features for better signal

#MedInc: skewed right (log-transform for better spread)
df["MedInc"] = np.log1p(df["MedInc"])

#New Col: Avg number of bedrooms per room (percent)
df["bedrooms_per_room"] = df["AveBedrms"] / df["AveRooms"] 

#df["bedrooms_per_room"].describe()

df["income_per_room"] = df["MedInc"] / df["AveRooms"]
df

In [ ]:
#LR Model 1

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd

X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']   #target

#train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .2, random_state = 42)  

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)  #training
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)

print(f"Avg House Price: {round(y_test.mean(), 2)}")
print(f"Mean Absolute Error: {round(mae, 2)}") # avg error 

r2 = r2_score(y_test, y_pred)
print(f"R-Squared Val: {round(r2, 2)}")

In [ ]:
df2 = df[df['MedHouseVal'] < 5]
df2

In [ ]:
#LR Model 2 (excluding MedHouseVal >= 5)

X_new = df2.drop(columns=['MedHouseVal'])
y_new = df2['MedHouseVal']   #target

X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X_new, y_new, test_size = .2, random_state = 42)  
 
scaler = StandardScaler()
X_train_scaled_new = scaler.fit_transform(X_train_new)
X_test_scaled_new = scaler.transform(X_test_new)

new_model = LinearRegression()
new_model.fit(X_train_scaled_new, y_train_new)  #training
y_pred_new = new_model.predict(X_test_scaled_new)

mae_new = mean_absolute_error(y_test_new, y_pred_new)

print(f"Avg House Price: {round(y_test_new.mean(), 2)}")
print(f"Mean Absolute Error: {round(mae_new, 2)}") # avg error in dollars

r2_new = r2_score(y_test_new, y_pred_new)
print(f"R-Squared Val: {round(r2_new, 2)}")

In [ ]:
from dataprep.eda import plot
import pandas as pd

results = pd.DataFrame({'Predicted': y_pred_new, 'Actual': y_test_new,})

#actual vs pred
plot(results, "Predicted House Price","Actual House Price ")

In [ ]:
#use original df without log-transformed MedInc and new cols

df = housing.frame

df["bedrooms_per_room"] = df["AveBedrms"] / df["AveRooms"] 

df["income_per_room"] = df["MedInc"] / df["AveRooms"]

df

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']   #target

#new train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .2, random_state = 42)  

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train) 

#model eval
rf_preds = rf_model.predict(X_test)
rf_r2 = r2_score(y_test, rf_preds)
rf_mae = mean_absolute_error(y_test, rf_preds)

print(f"Random Forest R2: {round(rf_r2, 4)}")
print(f"Random Forest Mean Abs Error: {round(rf_mae, 4)}")


In [ ]:
from dataprep.eda import plot
import pandas as pd

rf_results = pd.DataFrame({'Predicted House Price': rf_preds, 'Actual House Price': y_test})

plot(rf_results, "Predicted House Price","Actual House Price")